# Notebook 26 — Joint dissociated + spatial pathway scoring

`NicheformerEmbedder` produces a modality-invariant embedding for
both 10x dissociated scRNA-seq and Visium spatial measurements of the
same tissue. Passing those embeddings into F2's
`CrossPlatformAligner` (with modality labels instead of platform
labels) yields pathway scores in a joint frame.

Research use only. Not for clinical decision-making.

In [ ]:
import numpy as np
import pandas as pd

from pathway_subtyping.embed import NicheformerEmbedder, FallbackNicheformerEmbedder
from pathway_subtyping.harmonize import CrossPlatformAligner

rng = np.random.default_rng(0)
n_cells, n_genes = 150, 40
cluster_profiles = rng.standard_normal((3, n_genes)) * 1.5
cluster_ids = rng.integers(0, 3, size=n_cells)
base = cluster_profiles[cluster_ids]

gene_cols = [f'GENE_{i}' for i in range(n_genes)]
cell_idx = [f'cell_{i}' for i in range(n_cells)]
dissociated = pd.DataFrame(base + rng.normal(0, 0.3, base.shape), columns=gene_cols, index=cell_idx)
spatial     = pd.DataFrame(base + rng.normal(0, 0.3, base.shape), columns=gene_cols, index=cell_idx)

## 1. Embed both modalities in a shared basis

In [ ]:
embedder = NicheformerEmbedder(FallbackNicheformerEmbedder(embedding_dim=16))
diss_emb, spat_emb = embedder.embed_joint(dissociated, spatial)
print('diss shape:', diss_emb.embeddings.shape)
print('spat shape:', spat_emb.embeddings.shape)

## 2. Joint-frame alignment via F2's CrossPlatformAligner

Treat `dissociated` and `spatial` as "platforms" and let the existing
F2 aligner handle the modality shift.

In [ ]:
pathway_scores = pd.DataFrame(
    rng.standard_normal((n_cells, 5)),
    columns=[f'PATH_{i}' for i in range(5)],
    index=cell_idx,
)
all_scores = pd.concat([pathway_scores, pathway_scores + rng.normal(0, 0.5, (n_cells, 5))], axis=0).reset_index(drop=True)
modality   = ['dissociated'] * n_cells + ['spatial'] * n_cells
embeddings = np.vstack([diss_emb.embeddings, spat_emb.embeddings])

aligned = CrossPlatformAligner().fit_transform(all_scores, modality, embeddings)
aligned.aligned_scores.head()

## See also
- PSF v0.6 roadmap — Phase 3 F8: [docs/roadmap-v06-codeberg.md](../../docs/roadmap-v06-codeberg.md)
- F2 cross-platform harmonization guide: [docs/guides/cross-platform.md](../../docs/guides/cross-platform.md)